# CoT1.ipynb

Original Bi-CoT research notebook, recovered from the author’s AI_Study archive. Outputs, execution state, and machine metadata were removed; API credentials now come from `OPENAI_API_KEY`, and local paths now use `../data/workspace` relative to this notebook directory. See `docs/WORKFLOW.md` before selecting cells. Cells include experimental alternatives and data writes; this is not a single Run All pipeline.


In [ ]:
import os

def validate_sub_questions(example_dir: str):
    items = os.listdir(example_dir)
    
    # 0번 폴더, 1~N번 폴더 분리
    subq_dirs = [item for item in items if os.path.isdir(os.path.join(example_dir, item)) and item.isdigit()]
    subq_nums = sorted([int(s) for s in subq_dirs])
    
    # 0번(원질문) 폴더 분리
    has_zero = 0 in subq_nums
    if has_zero:
        subq_nums_wo_zero = [n for n in subq_nums if n != 0]
    else:
        subq_nums_wo_zero = subq_nums

    # 연속성: 1부터 시작
    if subq_nums_wo_zero:
        expected = list(range(1, len(subq_nums_wo_zero) + 1))
        if subq_nums_wo_zero != expected:
            print(f"\n❌ 오류: SubQ 폴더 번호가 연속되지 않거나 1부터 시작하지 않음.")
            print(f"    실제 번호: {subq_nums_wo_zero}")
            print(f"    기대 번호: {expected}")
        else:
            print(f"\n✅ SubQ 번호가 1부터 시작해 연속적입니다: {subq_nums_wo_zero}")
    else:
        print("❗ subQ 폴더(1 이상)가 없습니다.")

    # 각 subQ 폴더(1~N) 내부 0, 1 폴더 검사
    for subq_num in subq_nums_wo_zero:
        subq_path = os.path.join(example_dir, str(subq_num))
        expected_inner = ['0', '1']
        found_inner = [d for d in os.listdir(subq_path) if os.path.isdir(os.path.join(subq_path, d))]
        missing = [d for d in expected_inner if d not in found_inner]
        if missing:
            print(f"❌ 오류: subQ {subq_num} 폴더에 {missing} 폴더가 없습니다. (존재하는 폴더: {found_inner})")
        else:
            print(f"✅ subQ {subq_num} 폴더는 0, 1 폴더를 모두 포함합니다.")

    if has_zero:
        print(f"\n✅ 0번 폴더(원질문 폴더)는 존재하며, 내부 폴더는 검사하지 않습니다.")

    return subq_nums_wo_zero, len(subq_nums_wo_zero)

# 예시 경로
example_id_dir = "../data/workspace/testing/5a8b57f25542995d1e6f1371"
subqs, num_subqs = validate_sub_questions(example_id_dir)

print(f"\n총 SubQ 개수: {num_subqs}")

In [ ]:
from transformers import pipeline

qa = pipeline("text2text-generation", model="allenai/unifiedqa-t5-large", tokenizer="allenai/unifiedqa-t5-large")

In [ ]:
import json
from typing import List, Dict, Set

def extract_unique_title_sents_from_jsons(q_json_paths: List[str]) -> List[Dict[str, str]]:
    """
    여러 q.json 파일 경로를 받아, 모든 candidate_chains 내에서
    중복되지 않는 {"title": title, "sents": paragraph} 리스트를 반환
    """
    seen_paragraphs: Set[str] = set()
    result: List[Dict[str, str]] = []

    for q_json_path in q_json_paths:
        with open(q_json_path, "r", encoding="utf-8") as f:
            q_data = json.load(f)

        candidate_chains = q_data.get("candidate_chains", [])

        for chain in candidate_chains:
            for doc in chain:
                title = doc.get("title", "")
                sents = doc.get("sents", [])
                paragraph = " ".join(sents).strip()

                if paragraph and paragraph not in seen_paragraphs:
                    result.append({"title": title, "sents": paragraph})
                    seen_paragraphs.add(paragraph)

    return result

In [ ]:
import os
import json
from typing import List, Dict, Set

def extract_unique_title_sents_from_jsons(q_json_paths: List[str]) -> List[Dict[str, str]]:
    seen_paragraphs: Set[str] = set()
    result: List[Dict[str, str]] = []
    for q_json_path in q_json_paths:
        with open(q_json_path, "r", encoding="utf-8") as f:
            q_data = json.load(f)
        candidate_chains = q_data.get("candidate_chains", [])
        for chain in candidate_chains:
            for doc in chain:
                title = doc.get("title", "")
                sents = doc.get("sents", [])
                paragraph = " ".join(sents).strip()
                if paragraph and paragraph not in seen_paragraphs:
                    result.append({"title": title, "sents": paragraph})
                    seen_paragraphs.add(paragraph)
    return result

def qa(prompt, max_new_tokens=5, clean_up_tokenization_spaces=True):
    # 여기를 실제 QA 모델로 교체하세요
    return [{"generated_text": "yes"}]

def run_qa_and_save_answers(example_id_dir):
    # 모든 폴더(subq 및 0) 찾기
    sub_dirs = [d for d in os.listdir(example_id_dir)
                if os.path.isdir(os.path.join(example_id_dir, d)) and d.isdigit()]
    sub_dirs = sorted([int(d) for d in sub_dirs])

    for subq_num in sub_dirs:
        subq_path = os.path.join(example_id_dir, str(subq_num))
        
        # 0번 폴더(원질문)는 별도 처리: aug가 없음!
        if subq_num == 0:
            q_path = os.path.join(subq_path, "q.json")
            if not os.path.exists(q_path):
                print(f"❌ 경고: {q_path} 없음. 건너뜀.")
                continue
            # 단일 q.json만 사용
            title_sents_list = extract_unique_title_sents_from_jsons([q_path])
            context = "\n".join([item["sents"] for item in title_sents_list])
            with open(q_path, "r", encoding="utf-8") as f:
                q_data = json.load(f)
            question = q_data["question"]
            prompt = f"{question}\n{context}"
            result = qa(prompt, max_new_tokens=5, clean_up_tokenization_spaces=True)
            answer = result[0]['generated_text']
            print(f"[0] 🔎 Question: {question}")
            print(f"[0] 📄 Context:\n{context}")
            print(f"[0] ✅ Answer: {answer}")
            print("-" * 60)
            a_path = os.path.join(subq_path, "a.json")
            answer_obj = {
                "question": question,
                "context": context,
                "answer": answer,
                "context_detail": title_sents_list
            }
            with open(a_path, "w", encoding="utf-8") as f:
                json.dump(answer_obj, f, ensure_ascii=False, indent=2)
            continue  # 0번 끝!

        # 나머지 subq는 기존 방식 (aug 1,2)
        aug_q_json_paths = []
        aug_paths = []
        for aug in ['0', '1']:
            aug_path = os.path.join(subq_path, aug)
            q_path = os.path.join(aug_path, "q.json")
            if os.path.exists(q_path):
                aug_q_json_paths.append(q_path)
                aug_paths.append(aug_path)
            else:
                print(f"❌ 경고: {q_path} 없음. 건너뜀.")

        for aug_path, q_path in zip(aug_paths, aug_q_json_paths):
            # 이 aug의 q.json + 같은 subQ 내 다른 aug의 q.json
            other_q_jsons = [p for p in aug_q_json_paths if p != q_path]
            this_q_jsons = [q_path] + other_q_jsons
            title_sents_list = extract_unique_title_sents_from_jsons(this_q_jsons)
            context = "\n".join([item["sents"] for item in title_sents_list])

            with open(q_path, "r", encoding="utf-8") as f:
                q_data = json.load(f)
            question = q_data["question"]

            prompt = f"{question}\n{context}"
            result = qa(prompt, max_new_tokens=5, clean_up_tokenization_spaces=True)
            answer = result[0]['generated_text']

            print(f"[{subq_num}] 🔎 Question: {question}")
            print(f"[{subq_num}] 📄 Context:\n{context}")
            print(f"[{subq_num}] ✅ Answer: {answer}")
            print("-" * 60)

            a_path = os.path.join(aug_path, "a.json")
            answer_obj = {
                "question": question,
                "context": context,
                "answer": answer,
                "context_detail": title_sents_list
            }
            with open(a_path, "w", encoding="utf-8") as f:
                json.dump(answer_obj, f, ensure_ascii=False, indent=2)

# 실행 예시
example_id_dir = "../data/workspace/testing/5a8b57f25542995d1e6f1371"
run_qa_and_save_answers(example_id_dir)

In [ ]:
import os
import json
from typing import List, Dict, Set

def extract_unique_title_sents(title_sents_lists: List[List[Dict[str, str]]]) -> List[Dict[str, str]]:
    """여러 context_detail(list of dict)에서 중복 sents를 제거하며 합침."""
    seen_sents: Set[str] = set()
    result: List[Dict[str, str]] = []
    for title_sents in title_sents_lists:
        for item in title_sents:
            sents = item["sents"]
            if sents not in seen_sents:
                result.append(item)
                seen_sents.add(sents)
    return result

def qa(prompt, max_new_tokens=5, clean_up_tokenization_spaces=True):
    # 실제 QA모델로 교체!
    # 데모: yes/no 판별용
    if prompt.startswith('Is "'):
        # 예시: answer가 question에 substring으로 들어있으면 yes
        # 실제로는 모델에 따라 다름
        if prompt.split('"')[1] in prompt:
            return [{"generated_text": "yes"}]
        else:
            return [{"generated_text": "no"}]
    elif " or " in prompt:
        # 더미: a1 리턴
        return [{"generated_text": prompt.split(" or ")[0].strip()}]
    return [{"generated_text": "yes"}]

def process_subq(example_id_dir, subq_num):
    subq_path = os.path.join(example_id_dir, str(subq_num))
    aug_answers = []
    aug_context_details = []
    aug_contexts = []
    questions = []

    # 각 aug(0,1) a.json 읽기
    for aug in ['0', '1']:
        aug_path = os.path.join(subq_path, aug)
        a_path = os.path.join(aug_path, "a.json")
        if not os.path.exists(a_path):
            print(f"❌ {a_path} 없음. 건너뜀.")
            return
        with open(a_path, "r", encoding="utf-8") as f:
            a_data = json.load(f)
        aug_answers.append(a_data["answer"])
        aug_context_details.append(a_data["context_detail"])
        aug_contexts.append(a_data["context"])
        questions.append(a_data["question"])

    # 질문은 항상 1번 aug 폴더의 question
    question = questions[0]
    answer1 = aug_answers[0]
    answer2 = aug_answers[1]
    context_detail1 = aug_context_details[0]
    context_detail2 = aug_context_details[1]

    # 중복 없는 total_context_detail/total_context 생성
    total_context_detail = extract_unique_title_sents([context_detail1, context_detail2])
    total_context = "\n".join([item["sents"] for item in total_context_detail])

    cot_path = os.path.join(subq_path, "cot_1")
    os.makedirs(cot_path, exist_ok=True)
    a_cot_path = os.path.join(cot_path, "a.json")

    # 1. answer1 검증
    prompt1 = f'Is "{answer1}" the answer to "{question}"?'
    res1 = qa(prompt1 + "\n" + total_context)[0]['generated_text'].strip().lower().startswith("yes")
    # 2. answer2 검증
    prompt2 = f'Is "{answer2}" the answer to "{question}"?'
    res2 = qa(prompt2 + "\n" + total_context)[0]['generated_text'].strip().lower().startswith("yes")

    # 3. 판정
    if res1 and res2:
        # 둘 다 yes → or QA로 최종결정
        prompt = f'{answer1} or {answer2}, what is the answer of {question}?'
        chosen_answer = qa(prompt + "\n" + total_context)[0]['generated_text'].strip()
        if chosen_answer == answer1:
            final_context_detail = context_detail1
            final_context = aug_contexts[0]
        elif chosen_answer == answer2:
            final_context_detail = context_detail2
            final_context = aug_contexts[1]
        else:
            final_context_detail = total_context_detail
            final_context = total_context
    elif res1:
        # answer1만 yes
        chosen_answer = answer1
        final_context_detail = context_detail1
        final_context = aug_contexts[0]
    elif res2:
        # answer2만 yes
        chosen_answer = answer2
        final_context_detail = context_detail2
        final_context = aug_contexts[1]
    else:
        # 둘 다 yes 아니면 or QA
        prompt = f'{answer1} or {answer2}, what is the answer of {question}?'
        chosen_answer = qa(prompt + "\n" + total_context)[0]['generated_text'].strip()
        if chosen_answer == answer1:
            final_context_detail = context_detail1
            final_context = aug_contexts[0]
        elif chosen_answer == answer2:
            final_context_detail = context_detail2
            final_context = aug_contexts[1]
        else:
            final_context_detail = total_context_detail
            final_context = total_context

    answer_obj = {
        "question": question,
        "context": final_context,
        "answer": chosen_answer,
        "context_detail": final_context_detail
    }
    with open(a_cot_path, "w", encoding="utf-8") as f:
        json.dump(answer_obj, f, ensure_ascii=False, indent=2)

def run_cot_aggregation(example_id_dir):
    subq_dirs = [d for d in os.listdir(example_id_dir)
                 if os.path.isdir(os.path.join(example_id_dir, d)) and d.isdigit() and int(d) != 0]
    subq_dirs = sorted([int(d) for d in subq_dirs])
    for subq_num in subq_dirs:
        process_subq(example_id_dir, subq_num)

# 사용 예시
example_id_dir = "../data/workspace/testing/5a8b57f25542995d1e6f1371"
run_cot_aggregation(example_id_dir)

In [ ]:
from openai import OpenAI
import os
import json
from typing import List, Dict, Tuple, Set
import re

client = OpenAI(api_key=__import__("os").environ["OPENAI_API_KEY"])  # 본인 API 키 사용

def call_openai(prompt, model="gpt-4.1", max_tokens=256, temperature=0):
    print("\n======================")
    print("💬 [OpenAI Prompt]")
    print(prompt)
    print("======================")
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=temperature,
    )
    output = response.choices[0].message.content.strip()
    print("📝 [OpenAI Response]")
    print(output)
    print("======================\n")
    return output
def build_q_doc_pairs(subq_path, example_id_dir):
    """
    각 retrieval에 사용된 질문과 실제 retrieve된 문서(title, sents) 쌍을 모두 모은다.
    """
    q_doc_pairs = []

    # 원질문(0) 처리
    orig_a_path = os.path.join(example_id_dir, "0", "a.json")
    if os.path.exists(orig_a_path):
        with open(orig_a_path, "r", encoding="utf-8") as f:
            orig_a = json.load(f)
        orig_q = orig_a.get("retrieval_question", orig_a.get("question"))
        for doc in orig_a.get("context_detail", []):
            q_doc_pairs.append((orig_q, doc))

    # aug 0, 1 처리
    for aug in ['0', '1']:
        a_path = os.path.join(subq_path, aug, "a.json")
        if os.path.exists(a_path):
            with open(a_path, "r", encoding="utf-8") as f:
                a_data = json.load(f)
            aug_q = a_data.get("retrieval_question", a_data.get("question"))
            for doc in a_data.get("context_detail", []):
                q_doc_pairs.append((aug_q, doc))

    return q_doc_pairs


def merge_context_details(context_details_list: List[List[Dict[str, str]]]) -> List[Dict[str, str]]:
    """
    여러 context_detail (list of dict)에서 중복 sents를 제거하며 합침.
    """
    seen_sents: Set[str] = set()
    result: List[Dict[str, str]] = []
    for context_details in context_details_list:
        for item in context_details:
            sents = item["sents"]
            if sents not in seen_sents:
                result.append(item)
                seen_sents.add(sents)
    return result

def find_most_related_sentence_idx(answer: str, context: str) -> str:
    """
    context에서 answer와 가장 연관된(포함하는) 문장 번호(1-based)를 반환.
    없으면 'No'
    """
    sentences = [sent.strip() for sent in context.strip().split('\n') if sent.strip()]
    for idx, sent in enumerate(sentences, 1):
        if answer and answer in sent:
            return str(idx)
    return "No"

def is_answer_supported_by_context_return_idx(question: str, answer: str, context: str) -> Tuple[bool, str]:
    """
    1차 검증: Yes/No + (Yes라면 가장 연관된 문장 번호 반환)
    """
    prompt = (
        f'Below is a context consisting of multiple numbered passages, each with a title and its content.\n'
        f'Context:\n{context}\n\n'
        f'Question: Is "{answer}" the correct answer to "{question}"?\n'
        f'Answer only "Yes" or "No".\n'
        f'If your answer is "Yes", also return the single most influential sentence number (e.g., "Yes, 2").\n'
        f'If your answer is "No", just answer "No".'
    )
    out = call_openai(prompt).strip().lower()
    if out.startswith("yes"):
        idx = find_most_related_sentence_idx(answer, context)
        return True, idx
    else:
        return False, "No"

def get_freeform_answer_and_idx(question: str, context: str) -> Tuple[str, str]:
    """
    2차 검증: 주관식 답변 및 context에서 결정적 문장 번호 반환
    """
    prompt = (
        f'Below is a context consisting of multiple numbered passages, each with a title and its content.\n'
        f'Context:\n{context}\n\n'
        f'Question: What is the answer to "{question}"?\n'
        f'Read ALL context passages carefully and do not skip any.\n'
        f'If a correct answer exists in the context, return ONLY the answer (no explanation) followed by the single most relevant sentence number in parentheses, e.g., <answer> (number).\n'
        f'If NO answer can be found IN ANY of the context passages, reply exactly with "No answer found".\n'
        f'Do NOT answer with "not mentioned", "unknown", or anything else.\n'
        f'You must select from the given context. Only use the context above for your answer.'
    )
    out = call_openai(prompt).strip()
    # 예시: "Paris (2)" 또는 "yes (1)" 등 → answer, 번호 분리
    m = re.match(r"(.+)\((\d+)\)", out)
    if m:
        answer = m.group(1).strip()
        idx = m.group(2).strip()
        return answer, idx
    if out.lower() == "no answer found":
        return out, "No"
    # 혹시 번호 못 뽑으면 No 반환
    return out, "No"

def save_cot2_a_json(save_path: str, question: str, answer: str, evidence_title: str):
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    answer_obj = {
        "question": question,
        "answer": answer,
        "evidence_title": evidence_title
    }
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(answer_obj, f, ensure_ascii=False, indent=2)

def save_cot2_q_json(save_path: str, question: str):
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    question_obj = {"question": question}
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(question_obj, f, ensure_ascii=False, indent=2)

def reformulate_question_v2(
    original_question: str,
    q_doc_pairs: List[Tuple[str, Dict[str, str]]]
) -> str:
    prompt = (
        f"You are a search expert. Below, for each previously used retrieval question, you see the question and the document that was retrieved using that question.\n"
        f"For each question-document pair, briefly analyze why this question retrieved this document (e.g., which keywords or topic caused it), "
        f"and point out if the retrieved document is irrelevant or incomplete for answering the original main question.\n"
        f"After the analysis, based on all pairs, suggest a new, improved search question that:\n"
        f"  - Adds any missing keywords or concepts needed to retrieve more relevant documents for the main question\n"
        f"  - Avoids keywords or phrases that led to unwanted/irrelevant directions before\n"
        f"  - Is as specific as possible for the intended retrieval\n"
        f"\n"
        f"Main question:\n{original_question}\n\n"
        f"Previous retrievals:\n"
    )
    for (q, doc) in q_doc_pairs:
        prompt += (
            f"\n[Q] {q}\n"
            f"[Retrieved doc] {doc['title']}: {doc['sents']}\n"
        )
    prompt += (
        "\nAnalysis:\n"
        "- For each pair, state briefly what caused the retrieval and if it was relevant.\n"
        "\nNew, improved retrieval question (just the question, no explanation):"
    )
    return call_openai(prompt)

def process_subq_cot2(example_id_dir, subq_num):
    subq_path = os.path.join(example_id_dir, str(subq_num))
    cot_1_a_path = os.path.join(subq_path, "cot_1", "a.json")
    if not os.path.exists(cot_1_a_path):
        print(f"❌ cot_1/a.json 없음: {cot_1_a_path}")
        return
    with open(cot_1_a_path, "r", encoding="utf-8") as f:
        cot_1_a = json.load(f)
    question = cot_1_a["question"]
    answer = cot_1_a["answer"]
    context_items = cot_1_a["context_detail"]

    # 1차 context 생성 (번호+타이틀+문장, 중복 제거 필요 없음: cot_1 내에선 이미 unique로 만들어짐)
    context = "\n".join(
        [f"{i+1}.{item['title']}:{item['sents']}" for i, item in enumerate(context_items)]
    )

    cot_2_dir = os.path.join(subq_path, "cot_2")
    a_path = os.path.join(cot_2_dir, "a.json")
    q_path = os.path.join(cot_2_dir, "q.json")

    # 1차 검증
    print(f"\n====== [SubQ {subq_num}] 1차 검증 ======")
    is_yes, evidence_idx = is_answer_supported_by_context_return_idx(question, answer, context)
    if is_yes and evidence_idx != "No":
        try:
            evidence_title = context_items[int(evidence_idx)-1]["title"]
        except Exception:
            evidence_title = "No"
        save_cot2_a_json(a_path, question, answer, evidence_title)
        print(f"[{subq_num}] answer validated in cot_1 context, evidence title: {evidence_title}")
        return

    # 2차: 여러 context 합치기 + 중복 제거 + context 번호화
    context_details_list = []
    # cot_1 context_detail 포함
    if "context_detail" in cot_1_a:
        context_details_list.append(cot_1_a["context_detail"])
    for aug in ['0', '1']:
        a_path2 = os.path.join(subq_path, aug, "a.json")
        if os.path.exists(a_path2):
            with open(a_path2, "r", encoding="utf-8") as f:
                data = json.load(f)
            if "context_detail" in data:
                context_details_list.append(data["context_detail"])
    zero_a_path = os.path.join(example_id_dir, "0", "a.json")
    if os.path.exists(zero_a_path):
        with open(zero_a_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        if "context_detail" in data:
            context_details_list.append(data["context_detail"])
    # 중복 제거 병합!
    total_context_detail = merge_context_details(context_details_list)
    total_context = "\n".join(
        [f"{i+1}.{item['title']}:{item['sents']}" for i, item in enumerate(total_context_detail)]
    )

    print(f"\n====== [SubQ {subq_num}] 2차 검증 (aug+0+cot1, 중복제거) ======")
    answer2, evidence_idx2 = get_freeform_answer_and_idx(question, total_context)
    if evidence_idx2 != "No":
        try:
            evidence_title2 = total_context_detail[int(evidence_idx2)-1]["title"]
        except Exception:
            evidence_title2 = "No"
        save_cot2_a_json(a_path, question, answer2, evidence_title2)
        print(f"[{subq_num}] answer generated in aug+0+cot1 context, evidence title: {evidence_title2}")
        return

    # 3차: 질문 재생성 (이제는 retrieval_question + doc 쌍으로 넘김)
    print(f"\n====== [SubQ {subq_num}] 질문 재생성 단계 ======")
    q_doc_pairs = build_q_doc_pairs(subq_path, example_id_dir)
    new_question = reformulate_question_v2(question, q_doc_pairs)
    save_cot2_q_json(q_path, new_question)
    print(f"[{subq_num}] 질문 재구성 완료, cot_2/q.json에 저장됨.")

def run_cot2(example_id_dir):
    subq_dirs = [d for d in os.listdir(example_id_dir)
                 if os.path.isdir(os.path.join(example_id_dir, d)) and d.isdigit() and d != "0"]
    subq_dirs = sorted([int(d) for d in subq_dirs])
    for subq_num in subq_dirs:
        process_subq_cot2(example_id_dir, subq_num)

# 사용 예시
example_id_dir = "../data/workspace/testing/5a8b57f25542995d1e6f1371"
run_cot2(example_id_dir)